# Colab 3: Combined Two-Dataset Training

This notebook mounts Google Drive, pulls your project, configures combined training on two datasets, trains the model, zips the trained artifacts, and downloads the zip file.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Fill these in before running.
REPO_URL = 'https://github.com/your-username/your-repo.git'
BRANCH = 'main'
PROJECT_DIR = '/content/FND_2027'

# Example Drive dataset locations.
DATASET_DIR_1 = '/content/drive/MyDrive/FND_2027/ISOT Fake News Dataset'
DATASET_DIR_2 = '/content/drive/MyDrive/FND_2027/Fake News Dataset'

# Optional: folder in Drive where you also want the zip copied.
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/FND_2027_outputs'

In [ ]:
!rm -rf "$PROJECT_DIR"
!git clone --branch "$BRANCH" "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"

In [ ]:
%cd "$PROJECT_DIR"
!pip install -q -r requirements.txt

In [ ]:
%cd "$PROJECT_DIR"
import os
import yaml
from pathlib import Path

assert Path(DATASET_DIR_1).exists(), f'Missing dataset folder: {DATASET_DIR_1}'
assert Path(DATASET_DIR_2).exists(), f'Missing dataset folder: {DATASET_DIR_2}'

config_path = Path('config/config.yaml')
with config_path.open('r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

config['data']['dataset_name'] = 'generic'
config['data']['data_dir'] = DATASET_DIR_1
config['data']['dataset_names'] = ['generic', 'generic']
config['data']['data_dirs'] = [DATASET_DIR_1, DATASET_DIR_2]
config['logging']['checkpoint_dir'] = './checkpoints'
config['logging']['log_dir'] = './logs'
config['inference']['model_checkpoint'] = './checkpoints/best_model_combined.pt'

with config_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, sort_keys=False)

print('Updated config for combined training:')
print(config['data'])

In [ ]:
%cd "$PROJECT_DIR"
!python scripts/train.py --config config/config.yaml

In [ ]:
%cd "$PROJECT_DIR"
import shutil
from pathlib import Path

best_model = Path('checkpoints/best_model.pt')
combined_model = Path('checkpoints/best_model_combined.pt')
assert best_model.exists(), 'Training did not produce checkpoints/best_model.pt'
shutil.copy2(best_model, combined_model)
print(f'Saved combined checkpoint to: {combined_model}')

In [ ]:
%cd "$PROJECT_DIR"
!python scripts/evaluate.py --config config/config.yaml --checkpoint checkpoints/best_model_combined.pt

In [ ]:
%cd "$PROJECT_DIR"
import os
import shutil
from pathlib import Path

zip_base = Path('combined_model_artifacts')
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir='.', base_dir='checkpoints')
print(f'Created zip: {zip_path}')

# Append results and logs into the same zip using system zip.
!zip -ur combined_model_artifacts.zip results logs -x "*.ipynb_checkpoints*"

In [ ]:
%cd "$PROJECT_DIR"
import os
from pathlib import Path
from google.colab import files

zip_path = Path('combined_model_artifacts.zip')
assert zip_path.exists(), 'Zip file was not created.'

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
!cp combined_model_artifacts.zip "$DRIVE_OUTPUT_DIR/"
print(f'Copied zip to Drive: {DRIVE_OUTPUT_DIR}')

files.download(str(zip_path))